# Exposure analysis: data-transfer ON vs OFF

Looks at how bumblebee foraging changes between days when the data-transfer is on (250-300 Mb/s) and days when it's off. Designed to be run **after** `multi_day_pipeline.ipynb` has populated `data/multi_day/daily_summary.csv` and `per_track_indicators.csv`.

## Experimental design

Three-day-on, three-day-off cycle starting **2026-04-23**. Anything before that is treated as `BASELINE` (no transfer yet). The on-off-on-off design partially controls for monotonic time trends like temperature rising over the season.

| Period | Dates | Condition |
|---|---|---|
| Baseline | up to 2026-04-22 | BASELINE |
| Cycle 1 ON  | 2026-04-23 to 2026-04-25 | ON |
| Cycle 1 OFF | 2026-04-26 to 2026-04-28 | OFF |
| Cycle 2 ON  | 2026-04-29 to 2026-05-01 | ON |
| Cycle 2 OFF | 2026-05-02 to 2026-05-04 | OFF |
| ... | (3-on / 3-off continues) | |

## What this notebook answers

1. **Does total foraging volume change?** Daily exit + return counts under each condition.
2. **Do trips take longer or shorter?** Median trip duration ON vs OFF.
3. **Do bees navigate less efficiently?** Path tortuosity distribution per condition.
4. **Does activity timing shift?** Hourly daily-curve overlay.
5. **Are differences statistically defensible?** Mann-Whitney U tests.

## Caveats up front

- 3 days per condition per cycle is small. With one or two cycles you have ~6-12 (date, system) data points per condition. Treat large p-values as "we can't see an effect at this sample size", not "no effect exists".
- Confounds the design doesn't control for: weather (wind, rain, temperature, light), seasonal phenology of forage plants, hive maturation. Pull weather data (e.g. KNMI Wageningen) and overlay if effects look suggestive.
- A "treatment effect" that only shows on one camera is more likely a camera-specific artefact than biology. Always look at both `system 900` and `system 939` side-by-side.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


## 1. Treatment schedule

`condition_for(date)` -> `"BASELINE" | "ON" | "OFF"`. Edit the cycle anchor / length here if the schedule changes.


In [ ]:
CYCLE_ANCHOR = pd.Timestamp("2026-04-23")
CYCLE_ON_DAYS  = 3
CYCLE_OFF_DAYS = 3
CYCLE_LEN      = CYCLE_ON_DAYS + CYCLE_OFF_DAYS

def condition_for(date_like):
    """date_like: anything pd.Timestamp can parse (str, Timestamp, datetime)."""
    d = pd.Timestamp(date_like).normalize()
    if d < CYCLE_ANCHOR:
        return "BASELINE"
    days_since = (d - CYCLE_ANCHOR).days
    cycle_pos  = days_since % CYCLE_LEN
    return "ON" if cycle_pos < CYCLE_ON_DAYS else "OFF"


# Verify: print the schedule for the cached range.
print("Schedule preview:")
for d in pd.date_range("2026-04-19", "2026-05-08"):
    print(f"  {d.date()}  ->  {condition_for(d)}")


## 2. Load data


In [ ]:
DATA_DIR = Path("data/multi_day")
daily  = pd.read_csv(DATA_DIR / "daily_summary.csv")
tracks = pd.read_csv(DATA_DIR / "per_track_indicators.csv", parse_dates=["ts"])

daily["condition"]  = daily["date"].apply(condition_for)
tracks["condition"] = tracks["date"].apply(condition_for)

print("daily summary by condition:")
print(daily.groupby("condition").agg(
    n_pairs    = ("date",        "size"),
    n_dates    = ("date",        "nunique"),
    total_v3   = ("n_v3",        "sum"),
    total_ret  = ("n_returns",   "sum"),
    median_trip= ("median_trip_s","median"),
).to_string())

print(f"\nper_track rows: {len(tracks):,}  ({tracks['condition'].value_counts().to_dict()})")


## 3. Daily activity time series (ON-day shading)

Bar chart of `n_v3` and `n_returns` per day, with red shading for ON days. One panel per camera. The first thing to look at: do the red bars look systematically taller, shorter, or similar to the white-background bars?


In [ ]:
systems = sorted(daily["system_id"].unique())
fig, axes = plt.subplots(len(systems), 1, figsize=(13, 4 * len(systems)),
                          sharex=True, squeeze=False)

for ax, sys_id in zip(axes[:, 0], systems):
    sub = (daily[daily["system_id"] == sys_id]
           .sort_values("date").reset_index(drop=True))
    if sub.empty:
        ax.set_title(f"system {sys_id} - no data")
        continue
    x = pd.to_datetime(sub["date"])
    w = pd.Timedelta(hours=20)

    ax.bar(x, sub["n_v3"],      width=w, color="tab:blue",   alpha=0.85, label="v3 exits")
    ax.bar(x, sub["n_returns"], width=w * 0.55, color="tab:orange", alpha=0.95, label="returns")

    # Red shading on ON days, light grey on BASELINE.
    for _, row in sub.iterrows():
        d = pd.Timestamp(row["date"])
        if row["condition"] == "ON":
            ax.axvspan(d - pd.Timedelta(hours=12), d + pd.Timedelta(hours=12),
                       color="red", alpha=0.10, zorder=0)
        elif row["condition"] == "BASELINE":
            ax.axvspan(d - pd.Timedelta(hours=12), d + pd.Timedelta(hours=12),
                       color="grey", alpha=0.08, zorder=0)

    ax.set_title(f"system {sys_id}")
    ax.set_ylabel("count")
    ax.legend(loc="upper right")
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Daily exits & returns  -  red = transfer ON, grey = baseline", y=1.02)
plt.tight_layout()
plt.show()


## 4. ON vs OFF: box plots of daily metrics

Per-camera boxplots of the headline daily metrics. Overlapping boxes = no obvious effect. Cleanly separated boxes = look closer.


In [ ]:
METRICS  = [("n_v3",        "v3 exits / day"),
            ("n_returns",   "returns / day"),
            ("median_trip_s","median trip duration [s]")]
ORDER    = ["BASELINE", "OFF", "ON"]
COLORS   = {"BASELINE": "lightgrey", "OFF": "tab:green", "ON": "tab:red"}

fig, axes = plt.subplots(len(METRICS), len(systems),
                          figsize=(5 * len(systems), 3.5 * len(METRICS)),
                          squeeze=False)

for row, (col, label) in enumerate(METRICS):
    for c_ax, sys_id in enumerate(systems):
        ax = axes[row, c_ax]
        sub = daily[daily["system_id"] == sys_id]
        data = [sub.loc[sub["condition"] == cond, col].dropna() for cond in ORDER]
        bp = ax.boxplot(data, tick_labels=ORDER, patch_artist=True, widths=0.6)
        for patch, cond in zip(bp["boxes"], ORDER):
            patch.set_facecolor(COLORS[cond]); patch.set_alpha(0.7)
        # Overlay individual day points so n is honest.
        for i, d in enumerate(data):
            ax.scatter(np.full(len(d), i + 1) + np.random.uniform(-0.07, 0.07, len(d)),
                       d, color="black", s=15, alpha=0.6, zorder=3)
        ax.set_title(f"sys {sys_id} - {label}")
        ax.set_ylabel(label)
fig.suptitle("Daily metrics by condition (each black dot is one day)", y=1.01)
plt.tight_layout()
plt.show()


## 5. Path tortuosity per condition (per-track)

Distribution of `tortuosity = arc length / end-to-end displacement` across all flight tracks within each condition. Tortuosity = 1.0 means perfectly straight; higher = wigglier path. Heavier right-tail under ON would suggest bees navigating less directly when the transfer is on.

Tortuosity is clipped at 10 in the histogram so a few outliers don't flatten the rest. Statistical tests below use the full distribution.


In [ ]:
fig, axes = plt.subplots(1, len(systems), figsize=(7 * len(systems), 4),
                          sharey=True, squeeze=False)

for ax, sys_id in zip(axes[0], systems):
    sub = tracks[(tracks["system_id"] == sys_id) & tracks["tortuosity"].notna()]
    for cond, color in [("BASELINE", "grey"), ("OFF", "tab:green"), ("ON", "tab:red")]:
        vals = sub.loc[sub["condition"] == cond, "tortuosity"].clip(upper=10)
        if len(vals) == 0:
            continue
        ax.hist(vals, bins=50, alpha=0.45, color=color, density=True,
                label=f"{cond} (n={len(vals)})", edgecolor="black", linewidth=0.2)
    ax.set_xlabel("tortuosity (arc / displacement, clipped at 10)")
    ax.set_ylabel("density")
    ax.set_title(f"system {sys_id}")
    ax.legend()
fig.suptitle("Path tortuosity by condition", y=1.02)
plt.tight_layout()
plt.show()


## 6. Daily-curve overlay (ON vs OFF)

For each camera, overlay the average hourly exit-rate curve under ON days vs OFF days vs BASELINE. Tells you if the *timing* of foraging shifts even when total counts stay similar. If bees postpone or stretch their morning peak under ON, it shows up here as a phase-shifted curve.


In [ ]:
def hourly_mean_curve(df, system_id, cond):
    sub = df[(df["system_id"] == system_id) &
             (df["condition"] == cond) &
             (df["hive_exit_v3"] == True) &
             df["ts"].notna()].copy()
    if sub.empty:
        return None
    sub["hour"] = sub["ts"].dt.hour
    sub["date"] = sub["ts"].dt.date
    # Counts per (date, hour) -> then mean across dates -> stable hourly curve.
    counts = (sub.groupby(["date", "hour"]).size()
              .unstack(fill_value=0)
              .reindex(columns=range(24), fill_value=0))
    return counts.mean(axis=0)


fig, axes = plt.subplots(1, len(systems), figsize=(7 * len(systems), 4),
                          sharey=True, squeeze=False)

for ax, sys_id in zip(axes[0], systems):
    for cond, color in [("BASELINE", "grey"), ("OFF", "tab:green"), ("ON", "tab:red")]:
        curve = hourly_mean_curve(tracks, sys_id, cond)
        if curve is None:
            continue
        ax.plot(curve.index, curve.values, "o-", color=color, label=cond, linewidth=2)
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel("hour of day (Europe/Amsterdam)")
    ax.set_ylabel("mean v3 exits / hour")
    ax.set_title(f"system {sys_id}")
    ax.legend()
fig.suptitle("Average hourly exit-rate by condition", y=1.02)
plt.tight_layout()
plt.show()


## 7. Statistical tests

Two-sided Mann-Whitney U for each metric. Non-parametric because daily counts don't have to be Gaussian and we have small samples. Reports p-values per camera.

**How to read the p-values.** With 3-6 days per condition, anything p < 0.05 is plausible but not bulletproof - it's worth a closer look but not a publishable claim on its own. p > 0.20 means "we can't distinguish ON from OFF at this sample size". Add more days before drawing strong conclusions.


In [ ]:
from scipy import stats

print(f"{'metric':25s} {'system':>6s} {'ON med':>10s} {'OFF med':>10s} {'p (MWU)':>10s} {'n_on':>5s} {'n_off':>5s}")
print("-" * 78)
for sys_id in systems:
    sub = daily[daily["system_id"] == sys_id]
    for col, label in [("n_v3",         "n_v3"),
                       ("n_returns",    "n_returns"),
                       ("median_trip_s","median_trip_s")]:
        on  = sub.loc[sub["condition"] == "ON",  col].dropna()
        off = sub.loc[sub["condition"] == "OFF", col].dropna()
        if len(on) == 0 or len(off) == 0:
            print(f"{label:25s} {sys_id:6d} {'-':>10s} {'-':>10s} {'n/a':>10s} {len(on):>5d} {len(off):>5d}")
            continue
        u, p = stats.mannwhitneyu(on, off, alternative="two-sided")
        print(f"{label:25s} {sys_id:6d} {on.median():10.2f} {off.median():10.2f} {p:10.3f} {len(on):>5d} {len(off):>5d}")

# Tortuosity: per-track, MUCH more samples.
print()
for sys_id in systems:
    sub = tracks[(tracks["system_id"] == sys_id) & tracks["tortuosity"].notna()]
    on  = sub.loc[sub["condition"] == "ON",  "tortuosity"]
    off = sub.loc[sub["condition"] == "OFF", "tortuosity"]
    if len(on) == 0 or len(off) == 0:
        print(f"{'tortuosity (per track)':25s} {sys_id:6d} insufficient data")
        continue
    u, p = stats.mannwhitneyu(on, off, alternative="two-sided")
    print(f"{'tortuosity (per track)':25s} {sys_id:6d} {on.median():10.2f} {off.median():10.2f} {p:10.3f} {len(on):>5d} {len(off):>5d}")


## 8. What to do with the results

- **No clear effect.** Headline message: "in N days per condition we couldn't detect a change in foraging volume / trip duration / tortuosity." Note this isn't proof of no effect; it's a power-limited null result. Add more cycles.
- **Volume drops on ON days.** The simplest treatment effect interpretation: bees forage less when the transfer is on. Cross-check by overlaying weather - if ON days happened to be windier or colder you've measured weather, not transfer.
- **Volume unchanged but trips longer / more tortuous.** Subtler effect: same number of bees go out, but each trip is harder. More interesting biologically, points at navigation interference.
- **Effect on one camera only.** Most likely a camera-specific issue (calibration drift, vibration, jitter) rather than biology. Investigate the camera before drawing conclusions.

When you add new cached days, just re-run `multi_day_pipeline.ipynb` then re-run this notebook top-to-bottom. The condition tags update automatically.
